In [1]:
from classes.stock_client import StockClient
from classes.quant_analytics import QuantAnalytics as qa
import modules.plots as plots


In [2]:
client = StockClient(ttl_price=60, ttl_info=900)
quant = qa(client)

In [3]:
from modules.alerts import (AlertEngine, price_change, metric_threshold,
                             volatility_spike, new_52w_high_low,
                             drawdown_alert, kelly_edge_lost, ma_crossover_alert)

engine = AlertEngine(quant, client)

engine.add(price_change("NVDA", pct=0.05, direction="up"))
engine.add(price_change("AAPL", pct=0.05, direction="down"))
engine.add(metric_threshold("TSLA", "volatility", 0.80, "above"))
engine.add(metric_threshold("SPY",  "sharpe",     1.0,  "below"))
engine.add(volatility_spike("SPY", threshold=0.25))
engine.add(new_52w_high_low("MSFT"))
engine.add(drawdown_alert("QQQ", threshold=-0.15))
engine.add(kelly_edge_lost("NVDA"))
engine.add(ma_crossover_alert("SPY", fast=50, slow=200))

# run once
results = engine.run()

# run every 30 min in background
engine.schedule(interval_minutes=30)
engine.stop()

# add notifications
engine.set_log("alerts.jsonl")
engine.set_email(smtp_host="smtp.gmail.com", port=587,
                 user="you@gmail.com", password="...",
                 recipients=["you@gmail.com"])
engine.set_webhook("https://hooks.slack.com/services/...")

⚠️ [2026-04-24 14:06] Price ▲ 5% — NVDA | NVDA
   NVDA moved +5.11% today (gained $10.20)
   Value: 0.0511  |  Threshold: 0.0500

[14:06:18] Running 9 alerts...
Alert scheduler started — checking every 30 min. Call engine.stop() to halt.
[14:06:18] No alerts triggered (9 checked).
Alert scheduler stopped.


In [4]:
engine.stop()

Alert scheduler stopped.
